# Fine-tune InLegalBERT — Legal Contract Analyzer

Runs `pipeline/train_classifier_bert.py` from the repo on a free Colab GPU instead of the 12+ hour CPU estimate in the script's own docstring.

**Before running anything below:** `Runtime -> Change runtime type -> T4 GPU`, then `Save`.

What this notebook does:
1. Clones the repo's `claude/kind-newton-jsmo1r` branch (has the fine-tuning script + the fixed 47-class held-out evaluation).
2. Pulls `legal_contract_clauses.csv` (the CUAD training data) from the `main` branch of the same repo, since `pipeline/train_classifier.py` expects it one directory above the repo root.
3. Installs the handful of extra packages needed (Colab already ships torch + transformers).
4. Runs the fine-tune, which fine-tunes `law-ai/InLegalBERT` on the *exact* held-out split the TF-IDF+LR baseline was scored on, and writes `evaluation/results/inlegalbert_47class.json`.
5. Prints a baseline-vs-InLegalBERT comparison and downloads the results JSON.

Expect roughly 15-40 minutes on a T4 for 4 epochs over ~7,758 training rows (vs. 12+ hours on CPU).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU detected -- go to Runtime > Change runtime type > T4 GPU, then re-run this cell.'

In [ ]:
%cd /content
!rm -rf legal-contract-analyzer
!git clone -b claude/kind-newton-jsmo1r https://github.com/aniketh703/legal-contract-analyzer.git
%cd legal-contract-analyzer

In [ ]:
# The CUAD training CSV lives on the `main` branch (report-assets branch), not on
# the code branch. pipeline/train_classifier.py expects it one directory above
# the repo root (CSV_PATH = ROOT.parent / "legal_contract_clauses.csv"), i.e.
# /content/legal_contract_clauses.csv when this repo is cloned to /content/legal-contract-analyzer.
!git fetch origin main --depth 1
!git show origin/main:legal_contract_clauses.csv > ../legal_contract_clauses.csv
!wc -l ../legal_contract_clauses.csv

In [ ]:
# Minimal install -- skip requirements.txt's heavier/unrelated deps
# (label-studio, faiss-cpu, pdfplumber, pymupdf, pytesseract) that this
# training script doesn't touch and that would just slow the install down.
!pip install -q -U transformers accelerate scikit-learn joblib pandas numpy

In [ ]:
# Run as a module, not as a bare script path. train_classifier_bert.py lives
# inside the pipeline/ package itself, so `python pipeline/train_classifier_bert.py`
# makes Python treat pipeline/ as the run directory, which breaks resolving
# `pipeline` as a package (it's a namespace package -- no __init__.py) once code
# inside it does `from pipeline.X import Y`. `-m` anchors sys.path to the cwd
# (the repo root) instead, which resolves it correctly.
!python -m pipeline.train_classifier_bert

In [ ]:
import json

with open("evaluation/results/tfidf_lr_baseline_47class.json") as f:
    baseline = json.load(f)
with open("evaluation/results/inlegalbert_47class.json") as f:
    bert = json.load(f)

b_cr = baseline["classification_report"]
n_cr = bert["classification_report"]

print(f"{'Metric':<22}{'TF-IDF + LR':>14}{'InLegalBERT':>14}")
print(f"{'Accuracy':<22}{baseline['accuracy']:>14.4f}{bert['accuracy']:>14.4f}")
print(f"{'Macro F1':<22}{b_cr['macro avg']['f1-score']:>14.4f}{n_cr['macro avg']['f1-score']:>14.4f}")
print(f"{'Weighted F1':<22}{b_cr['weighted avg']['f1-score']:>14.4f}{n_cr['weighted avg']['f1-score']:>14.4f}")

In [ ]:
from google.colab import files

files.download("evaluation/results/inlegalbert_47class.json")

## Optional: keep the fine-tuned model weights

Only run this if you want to reuse the fine-tuned model later (e.g. to wire it into the live demo) -- it's a few hundred MB, so skip it if you only need the numbers above.

In [ ]:
!zip -qr inlegalbert_classifier.zip models/inlegalbert_classifier
from google.colab import files

files.download("inlegalbert_classifier.zip")